# 046번 공감형 대화 데이터셋 탐색

목적: JSON 내부 구조 파악 → 이후 few-shot 추출 스크립트 작성 기반 마련  
대상: `Validation/02.라벨링데이터/VL_슬픔_부모자녀,조손.zip` 하나만 먼저 확인

## Cell 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2. 경로 설정

In [ ]:
import os

# --- Step 1. MyDrive 하위에서 Dadam_dataSet 폴더 찾기 ---
MYDRIVE = '/content/drive/MyDrive'

def find_dir(root, target_name, max_depth=4):
    """root 아래에서 target_name이 포함된 폴더를 재귀 탐색"""
    for dirpath, dirnames, _ in os.walk(root):
        depth = dirpath.replace(root, '').count(os.sep)
        if depth >= max_depth:
            del dirnames[:]
            continue
        for d in dirnames:
            if target_name in d:
                return os.path.join(dirpath, d)
    return None

dataset_root = find_dir(MYDRIVE, 'Dadam_dataSet')
print('데이터셋 루트:', dataset_root)

# --- Step 2. 046번 폴더 탐색 ---
search_root = dataset_root if dataset_root else MYDRIVE
folder_046 = find_dir(search_root, '046')
print('046 폴더:', folder_046)

# --- Step 3. 046번 폴더 하위 구조 출력 ---
if folder_046:
    for dirpath, dirnames, filenames in os.walk(folder_046):
        depth = dirpath.replace(folder_046, '').count(os.sep)
        if depth > 4:
            continue
        indent = '  ' * depth
        print(f'{indent}{os.path.basename(dirpath)}/')
        for fname in filenames[:3]:  # 파일은 3개만 미리보기
            print(f'{indent}  {fname}')
        if len(filenames) > 3:
            print(f'{indent}  ... 외 {len(filenames)-3}개')
else:
    # 폴더를 못 찾은 경우 MyDrive 최상위 목록 출력
    print('\n[폴더를 찾지 못했습니다] MyDrive 최상위 목록:')
    for name in sorted(os.listdir(MYDRIVE)):
        print(' ', name)

## Cell 3. zip 내부 파일 목록 확인

In [ ]:
import zipfile

# Cell 2에서 확인된 경로 기반으로 설정
BASE_PATH = '/content/drive/MyDrive/Dadam_dataSet/046.공감형 대화/data/Validation/02.라벨링데이터'

# 어르신 정서에 가장 관련 높은 파일 우선 선택
SAMPLE_ZIP = os.path.join(BASE_PATH, 'VL_슬픔_부모자녀,조손.zip')

print('경로 존재 여부:', os.path.exists(SAMPLE_ZIP))

with zipfile.ZipFile(SAMPLE_ZIP, 'r') as z:
    all_files = z.namelist()

print(f'파일 총 {len(all_files)}개')
print('--- 처음 10개 ---')
for f in all_files[:10]:
    print(f)

## Cell 4. JSON 파일 하나 열어서 구조 확인

In [ ]:
import json

# JSON 구조 확인 완료:
# utterances[] 배열에서 role="speaker"(사람), role="listener"(AI 공감 응답) 쌍
# info.speaker_emotion: 감정, info.relation: 관계, info.listener_behavior: 응답 유형

with zipfile.ZipFile(SAMPLE_ZIP, 'r') as z:
    json_files = [f for f in z.namelist() if f.endswith('.json')]
    first_file = json_files[0]
    with z.open(first_file) as f:
        data = json.load(f)

print(f'파일명: {first_file}')
print(f'감정: {data["info"]["speaker_emotion"]}')
print(f'관계: {data["info"]["relation"]}')
print(f'응답 유형: {data["info"]["listener_behavior"]}')
print(f'발화 수: {len(data["utterances"])}개')
print(f'JSON 파일 총 {len(json_files)}개')

## Cell 5. 최상위 키 목록 및 중첩 구조 요약

어떤 필드명으로 발화자/응답이 구분되는지 파악

In [ ]:
# 발화 쌍(speaker→listener) 연속 턴 추출 함수
def extract_pairs(utterances):
    """speaker 발화 바로 다음 listener 응답을 쌍으로 묶어 반환"""
    pairs = []
    for i in range(len(utterances) - 1):
        curr = utterances[i]
        nxt = utterances[i + 1]
        if curr['role'] == 'speaker' and nxt['role'] == 'listener':
            pairs.append({
                'speaker': curr['text'],
                'listener': nxt['text'],
                'empathy': nxt.get('listener_empathy') or []
            })
    return pairs

# 샘플 파일에서 쌍 추출 미리보기
pairs = extract_pairs(data['utterances'])
print(f'추출된 쌍: {len(pairs)}개')
for i, p in enumerate(pairs):
    print(f'\n--- 쌍 {i+1} | 공감유형: {p["empathy"]} ---')
    print(f'사람 : {p["speaker"]}')
    print(f'AI   : {p["listener"]}')

## Cell 6. 발화 쌍 미리보기 (3개)

구조 확인 후 필드명을 아래에 채워서 실행  
예) `talk → content → HS01` (사람 발화), `SS01` (시스템 응답)

In [ ]:
import pandas as pd

# extract_pairs 함수 재정의 (독립 실행 가능하도록)
def extract_pairs(utterances):
    """speaker 발화 바로 다음 listener 응답을 쌍으로 묶어 반환"""
    pairs = []
    for i in range(len(utterances) - 1):
        curr = utterances[i]
        nxt = utterances[i + 1]
        if curr['role'] == 'speaker' and nxt['role'] == 'listener':
            pairs.append({
                'speaker': curr['text'],
                'listener': nxt['text'],
                'empathy': nxt.get('listener_empathy') or []
            })
    return pairs

# 6감정 전체 × 어르신 관련 관계 조합 (실제 파일명 기준)
TARGET_ZIPS = [
    # 슬픔
    'VL_슬픔_부모자녀,조손.zip',
    'VL_슬픔_친구.zip',
    'VL_슬픔_지인.zip',
    # 상처
    'VL_상처_부모자녀,조손.zip',
    'VL_상처_친구.zip',
    'VL_상처_지인.zip',
    # 기쁨
    'VL_기쁨_부모자녀,조손.zip',
    'VL_기쁨_친구.zip',
    'VL_기쁨_지인.zip',
    # 당황
    'VL_당황_부모자녀,조손.zip',
    'VL_당황_친구.zip',
    'VL_당황_지인.zip',
    # 분노
    'VL_분노_부모자녀,조손.zip',
    'VL_분노_친구.zip',
    'VL_분노_지인.zip',
    # 불안
    'VL_불안_부모자녀,조손.zip',
    'VL_불안_친구.zip',
    'VL_불안_지인.zip',
]

all_pairs = []
missing = []

for zip_name in TARGET_ZIPS:
    zip_path = os.path.join(BASE_PATH, zip_name)
    if not os.path.exists(zip_path):
        missing.append(zip_name)
        continue

    with zipfile.ZipFile(zip_path, 'r') as z:
        json_files = [f for f in z.namelist() if f.endswith('.json')]
        for fname in json_files:
            with z.open(fname) as f:
                d = json.load(f)
            info = d['info']
            for pair in extract_pairs(d['utterances']):
                all_pairs.append({
                    'emotion': info['speaker_emotion'],
                    'relation': info['relation'],
                    'speaker_relation': info.get('speaker_relation', ''),
                    'empathy_type': ', '.join(pair['empathy']),
                    'speaker': pair['speaker'],
                    'listener': pair['listener'],
                    'avg_rating': info['evaluation']['avg_rating'],
                    'source_file': fname,
                })

if missing:
    print(f'[없는 파일 {len(missing)}개]')
    for m in missing:
        print(f'  {m}')
else:
    print('모든 파일 정상 로드')

df = pd.DataFrame(all_pairs)
print(f'\n총 추출 쌍: {len(df)}개')
print('\n[감정 분포]')
print(df['emotion'].value_counts())
print('\n[관계 분포]')
print(df['relation'].value_counts())
print('\n[발화자 역할 분포]')
print(df['speaker_relation'].value_counts())

In [ ]:
# 품질 필터
# 1. avg_rating 5.0 (최고 품질만)
# 2. 발화 길이 15자 이상 (의미 있는 발화)
# 3. 위로·동조 공감 유형 우선
# 4. 어르신(부모/조부모) 입장 화자만
SENIOR_ROLES = ['아버지, 어머니', '할아버지, 할머니']

df_filtered = df[
    (df['avg_rating'] == 5.0) &
    (df['speaker'].str.len() >= 15) &
    (df['listener'].str.len() >= 15) &
    (df['empathy_type'].str.contains('위로|동조', na=False)) &
    (df['speaker_relation'].isin(SENIOR_ROLES))
].copy()

print(f'필터 후 쌍: {len(df_filtered)}개')
print('\n[감정별 분포]')
print(df_filtered['emotion'].value_counts())

print('\n[미리보기 — 감정별 1개씩]')
for emotion in df_filtered['emotion'].unique():
    row = df_filtered[df_filtered['emotion'] == emotion].iloc[0]
    print(f'\n감정: {row["emotion"]} | 관계: {row["relation"]} | 공감: {row["empathy_type"]}')
    print(f'어르신: {row["speaker"]}')
    print(f'AI    : {row["listener"]}')

In [ ]:
# 결과 CSV로 저장 (Drive에 저장 → 수동 선별 후 프롬프트에 삽입)
OUTPUT_PATH = '/content/drive/MyDrive/Dadam_dataSet/046_fewshot_candidates.csv'
df_filtered.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUTPUT_PATH}')
print(f'총 {len(df_filtered)}개 후보 → 이 중 5~10개를 수동으로 골라 프롬프트에 사용')

## Cell 9. AI 말동무 말투로 변환 (GPT 활용)

현재 listener 응답이 "자녀 말투"로 되어 있음 → AI 말동무 말투로 변환 필요

- **Before**: "아버지, 저도 안타깝네요. 기운 내세요."
- **After**: "많이 힘드셨겠어요. 그런 마음이 드실 만해요."

In [ ]:
import openai
import time

# OpenAI API 키 설정
OPENAI_API_KEY = ''  # 여기에 입력하거나 아래 주석 해제 후 Colab Secret 사용
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = openai.OpenAI(api_key=OPENAI_API_KEY)

CONVERT_SYSTEM_PROMPT = """당신은 AI 말동무 응답 변환 전문가입니다.

주어진 응답은 자녀가 부모에게 하는 말투로 작성되어 있습니다.
이것을 어르신과 대화하는 따뜻한 AI 말동무 말투로 변환해주세요.

변환 규칙:
- "아버지/어머니/할아버지/할머니" 같은 호칭 제거
- "저도", "우리 가족" 등 자녀 관점 표현 제거
- "많이 힘드셨겠어요", "그러셨군요", "그런 마음이 드실 만해요" 같은 공감 말투 사용
- 1~2문장으로 간결하게
- 반말 금지, 존댓말 유지
- 변환된 응답 텍스트만 출력 (설명 없이)"""

def convert_to_ai_tone(listener_text):
    """자녀 말투 → AI 말동무 말투 변환"""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'system', 'content': CONVERT_SYSTEM_PROMPT},
            {'role': 'user', 'content': listener_text}
        ],
        temperature=0.3,
        max_tokens=200,
    )
    return response.choices[0].message.content.strip()

# 42개 전체 변환 (API 호출 간격 0.5초)
print('변환 시작...')
converted = []
for i, row in df_filtered.iterrows():
    result = convert_to_ai_tone(row['listener'])
    converted.append(result)
    if (len(converted)) % 10 == 0:
        print(f'  {len(converted)}/{len(df_filtered)} 완료')
    time.sleep(0.5)

df_filtered = df_filtered.copy()
df_filtered['listener_converted'] = converted
print(f'\n변환 완료: {len(df_filtered)}개')

# 변환 전후 비교 미리보기
for _, row in df_filtered.head(3).iterrows():
    print(f'\n어르신 : {row["speaker"]}')
    print(f'원본   : {row["listener"]}')
    print(f'변환   : {row["listener_converted"]}')

In [ ]:
# 변환 결과 CSV로 최종 저장
OUTPUT_FINAL = '/content/drive/MyDrive/Dadam_dataSet/046_fewshot_converted.csv'
df_filtered.to_csv(OUTPUT_FINAL, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUTPUT_FINAL}')
print(f'총 {len(df_filtered)}개 → 스프레드시트로 열어서 5~8개 수동 선별 후 프롬프트에 삽입')

## Cell 11. 감정별 최적 예시 선별 → few-shot JSON 저장

선별 기준:
- 감정별 2개씩 (6감정 × 2 = 12개)
- `listener_converted` 길이가 너무 길지 않은 것 (100자 이하)
- `speaker` 발화가 어르신 정서를 잘 드러내는 것
- 중복 주제 제외

In [ ]:
import json

# 변환된 CSV 다시 로드 (독립 실행용)
df_converted = pd.read_csv(OUTPUT_FINAL, encoding='utf-8-sig')

# 선별 기준
# 1. listener_converted 길이 25자 이상 100자 이하 (너무 짧거나 긴 것 제외)
# 2. speaker 길이 20자 이상 (의미 있는 발화)
# 3. listener_converted에 호칭(아버지/어머니 등) 잔존 여부 필터
HONORIFICS = ['아버지', '어머니', '할아버지', '할머니', '엄마', '아빠']

df_pool = df_converted[
    (df_converted['listener_converted'].str.len() >= 25) &
    (df_converted['listener_converted'].str.len() <= 100) &
    (df_converted['speaker'].str.len() >= 20) &
    (~df_converted['listener_converted'].str.contains('|'.join(HONORIFICS), na=False))
].copy()

print(f'선별 풀: {len(df_pool)}개')
print(df_pool['emotion'].value_counts())

# 감정별 2개씩 선별 (speaker 길이 기준 상위 → 풍부한 발화 우선)
selected_rows = []
for emotion in ['슬픔', '기쁨', '당황', '분노', '불안', '상처']:
    pool = df_pool[df_pool['emotion'] == emotion].copy()
    pool = pool.sort_values('speaker', key=lambda x: x.str.len(), ascending=False)
    picked = pool.head(2)
    selected_rows.append(picked)
    print(f'{emotion}: {len(picked)}개 선별')

df_selected = pd.concat(selected_rows, ignore_index=True)

# few-shot JSON 형식으로 변환
fewshot = []
for _, row in df_selected.iterrows():
    fewshot.append({
        'emotion': row['emotion'],
        'relation': row['relation'],
        'human': row['speaker'],
        'assistant': row['listener_converted']
    })

print(f'\n최종 선별: {len(fewshot)}개')
for item in fewshot:
    print(f'\n[{item["emotion"]}]')
    print(f'어르신: {item["human"]}')
    print(f'AI    : {item["assistant"]}')

In [ ]:
# Drive와 로컬 다운로드 경로 두 곳에 저장
OUTPUT_FEWSHOT_DRIVE = '/content/drive/MyDrive/Dadam_dataSet/046_fewshot_final.json'

with open(OUTPUT_FEWSHOT_DRIVE, 'w', encoding='utf-8') as f:
    json.dump(fewshot, f, ensure_ascii=False, indent=2)

print(f'저장 완료: {OUTPUT_FEWSHOT_DRIVE}')
print(f'총 {len(fewshot)}개 few-shot 예시 → TASK-02 voice-chat 프롬프트에 삽입 예정')